# 4. Evaluation and comparison

Run notebooks 01, 02, and 03 first. This notebook compares **all 24 runs using validation metrics**, fixes one validation winner per family, then evaluates all 24 runs on the test set for the experiment report. Selection is fixed before loading test labels. It produces:

- Accuracy; precision, recall, and F1 for each class; macro precision, recall, and F1; and weighted scores.
- Count and row-normalized confusion matrices (rows = true classes; columns = predicted classes).
- Training/validation loss and accuracy curves for every run.
- A controlled comparison of embedding type, dimension, and model configuration using validation data.
- A reference table of all ten neural-network accuracies in the paper, without claiming exact numerical reproduction.

The test evaluations are for reporting, not for choosing a new configuration. **Do not tune settings after examining test results.** Set `EVALUATE_ALL_RUNS = False` below if you only want the two validation winners evaluated; the validation-based winners stay fixed. One seed is used; repeated runs would be needed to quantify uncertainty.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)
from IPython.display import display

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
RESULTS = ROOT / "results"
REPORTS = RESULTS / "evaluation"
REPORTS.mkdir(parents=True, exist_ok=True)
metadata = json.loads((ARTIFACTS / "metadata.json").read_text())
CLASS_NAMES = metadata["class_names"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
with np.load(ARTIFACTS / "embeddings.npz") as saved:
    embeddings = {name: saved[name] for name in saved.files}

summaries = []
for family in ["feedforward", "lstm"]:
    path = RESULTS / family / "validation_summary.csv"
    assert path.exists(), f"Missing {family} results. Run notebooks 02 and 03 first."
    table = pd.read_csv(path)
    assert len(table) == 12, f"{family}: expected 12 completed runs; finish its training notebook."
    assert table["preparation_id"].eq(metadata["preparation_id"]).all(), "Preparation changed; retrain."
    summaries.append(table)
comparison = pd.concat(summaries, ignore_index=True)
comparison = comparison.sort_values(["val_macro_f1", "val_loss", "run_id"],
                                     ascending=[False, True, True])
display(comparison.drop(columns=["preparation_id"]))
comparison.to_csv(REPORTS / "all_validation_results.csv", index=False)


## Compare the experiments on validation data

The table below averages over configurations to summarize embedding effects. The controlled table holds configuration fixed so embedding comparisons are easier to interpret. Scores are measured; no paper scores are substituted for model outputs.


In [ ]:
embedding_effect = comparison.pivot_table(index="embedding", columns="family",
                                          values=["val_accuracy", "val_macro_f1"], aggfunc="mean")
display(embedding_effect)
controlled = comparison.pivot_table(index=["family", "config"], columns="embedding",
                                    values="val_macro_f1")
display(controlled)
controlled.to_csv(REPORTS / "controlled_validation_macro_f1.csv")
configuration_effect = comparison.groupby(["family", "config"])[
    ["val_accuracy", "val_macro_f1"]].mean()
display(configuration_effect)

fig, ax = plt.subplots(figsize=(10, 4))
comparison.groupby(["family", "embedding"])["val_macro_f1"].mean().unstack(0).plot.bar(ax=ax)
ax.set(ylabel="Mean validation macro F1", xlabel="Embedding",
       title="Embedding comparison (average across each family's configurations)", ylim=(0, 1))
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(REPORTS / "validation_embedding_comparison.png", dpi=160)
plt.show()

# First in each family after the common sort: validation winner.
winners = comparison.groupby("family", sort=False).head(1).copy()
display(winners[["family", "run_id", "val_accuracy", "val_macro_f1", "best_epoch"]])
winners.to_csv(REPORTS / "selected_validation_winners.csv", index=False)


## Plot and save learning curves for all runs


In [ ]:
CURVES = REPORTS / "learning_curves"
CURVES.mkdir(exist_ok=True)
for family in ["feedforward", "lstm"]:
    runs = comparison[comparison["family"] == family]
    fig, axes = plt.subplots(len(runs), 2, figsize=(11, 3 * len(runs)), squeeze=False)
    for row, run in enumerate(runs.itertuples()):
        history = pd.read_csv(RESULTS / family / f"{run.run_id}_history.csv")
        for col, metric in enumerate(["loss", "accuracy"]):
            ax = axes[row, col]
            ax.plot(history["epoch"], history[f"train_{metric}"], label="Training")
            ax.plot(history["epoch"], history[f"val_{metric}"], label="Validation")
            ax.axvline(run.best_epoch, color="gray", linestyle=":", label="Selected epoch")
            ax.set(title=f"{run.run_id}: {metric}", xlabel="Epoch", ylabel=metric)
            ax.legend(fontsize=8)
            ax.grid(alpha=0.2)
    fig.suptitle(f"{family}: learning curves for all runs", y=1.001)
    fig.tight_layout()
    fig.savefig(CURVES / f"{family}_all_curves.png", dpi=140, bbox_inches="tight")
    plt.show()


## Restore the models in a fresh kernel

These are the same short model definitions used for training. Their saved settings restore dropout and embedding behavior. Checkpoints contain tensors only and are loaded with `weights_only=True`.


In [ ]:
# These globals are replaced by the saved settings before constructing each model.
TRAINABLE_EMBEDDINGS = False
DROPOUT = 0.2


def log_probabilities(logits, output_activation):
    if output_activation == "softmax":
        return torch.log_softmax(logits, dim=1)
    log_scores = torch.nn.functional.logsigmoid(logits)
    return log_scores - torch.logsumexp(log_scores, dim=1, keepdim=True)


In [ ]:
class FeedforwardNN(nn.Module):
    def __init__(self, matrix, config):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(matrix), freeze=not TRAINABLE_EMBEDDINGS, padding_idx=0)
        layers = []
        width = matrix.shape[1]
        for hidden_width, activation in zip(config["hidden_sizes"], config["activations"]):
            layers.append(nn.Linear(width, hidden_width))
            layers.append(nn.ReLU() if activation == "relu" else nn.Sigmoid())
            layers.append(nn.Dropout(DROPOUT))
            width = hidden_width
        layers.append(nn.Linear(width, 3))
        self.classifier = nn.Sequential(*layers)

    def forward(self, X):
        mask = (X != 0).unsqueeze(-1)
        vectors = self.embedding(X)
        pooled = (vectors * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return log_probabilities(self.classifier(pooled), "softmax")


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, matrix, config):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(matrix), freeze=not TRAINABLE_EMBEDDINGS, padding_idx=0)
        self.lstm = nn.LSTM(matrix.shape[1], config["hidden_size"], batch_first=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(config["hidden_size"], 3)
        self.output_activation = config["output_activation"]

    def forward(self, X):
        lengths = (X != 0).sum(dim=1).clamp(min=1).cpu()
        vectors = self.embedding(X)
        packed = nn.utils.rnn.pack_padded_sequence(
            vectors, lengths, batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        logits = self.classifier(self.dropout(hidden[-1]))
        return log_probabilities(logits, self.output_activation)


## Final test evaluation

All-class precision, recall, and F1 are computed with the fixed label order `[0, 1, 2]`. Macro F1 gives each class equal weight. `zero_division=0` handles a class with no predicted examples. Prediction files include the original text, true label, predicted label, and three class probabilities.


In [ ]:
EVALUATE_ALL_RUNS = True  # False evaluates only the two validation winners
selected = comparison.copy() if EVALUATE_ALL_RUNS else winners.copy()

with np.load(ARTIFACTS / "encoded_data.npz") as saved:
    X_test, y_test = saved["X_test"], saved["y_test"]
# Text copies are not needed: row order is preserved in the saved arrays.
# Recover texts by source location or the portable data/ folder, verifying the file.
import hashlib
source_dir = Path("/Users/arvind/Documents/Sem5/ML/Mini Project")
if not (source_dir / "test.csv").exists():
    source_dir = ROOT / "data"
test_path = source_dir / "test.csv"
assert hashlib.sha256(test_path.read_bytes()).hexdigest() == metadata["csv_sha256"]["test"], \
    "test.csv changed since preparation. Rerun preparation and training."
test_text = pd.read_csv(test_path)["comment_text"].astype(str).to_numpy()
assert len(test_text) == len(y_test)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.long)),
                         batch_size=256, shuffle=False)
test_rows = []
for run in selected.itertuples():
    directory = RESULTS / run.family
    settings = json.loads((directory / f"{run.run_id}_settings.json").read_text())
    assert settings["preparation_id"] == metadata["preparation_id"]
    TRAINABLE_EMBEDDINGS = settings["trainable_embeddings"]
    DROPOUT = settings["dropout"]
    model_class = FeedforwardNN if run.family == "feedforward" else LSTMClassifier
    model = model_class(embeddings[run.embedding], settings["config"]).to(DEVICE)
    state = torch.load(directory / f"{run.run_id}.pt", map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    model.eval()
    probabilities = []
    with torch.no_grad():
        for (X_batch,) in test_loader:
            probabilities.append(model(X_batch.to(DEVICE)).exp().cpu().numpy())
    probabilities = np.concatenate(probabilities)
    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    predictions = probabilities.argmax(axis=1)
    report = classification_report(y_test, predictions, labels=[0, 1, 2],
                                   target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    stem = f"{run.family}__{run.run_id}"
    report_df.to_csv(REPORTS / f"{stem}_classification_report.csv")
    print(f"\n{run.family}: {run.run_id}")
    display(report_df)
    test_rows.append({"family": run.family, "run_id": run.run_id,
                      "embedding": run.embedding, "config": run.config,
                      "val_macro_f1": run.val_macro_f1,
                      "test_accuracy": accuracy_score(y_test, predictions),
                      "test_macro_precision": report["macro avg"]["precision"],
                      "test_macro_recall": report["macro avg"]["recall"],
                      "test_macro_f1": report["macro avg"]["f1-score"]})
    output = pd.DataFrame({"comment_text": test_text, "true_label": y_test,
                           "predicted_label": predictions})
    for label in [0, 1, 2]:
        output[f"probability_class_{label}"] = probabilities[:, label]
    output.to_csv(REPORTS / f"{stem}_predictions.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, normalization, title in [(axes[0], None, "Counts"),
                                      (axes[1], "true", "Normalized by true class")]:
        matrix = confusion_matrix(y_test, predictions, labels=[0, 1, 2], normalize=normalization)
        ConfusionMatrixDisplay(matrix, display_labels=[0, 1, 2]).plot(
            ax=ax, cmap="Blues", values_format="d" if normalization is None else ".2f",
            colorbar=False)
        ax.set_title(title)
        pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
            REPORTS / f"{stem}_confusion_{'counts' if normalization is None else 'normalized'}.csv")
    fig.suptitle(f"{run.family}: {run.run_id}; 0=neutral/positive, 1=negative, 2=derogatory")
    fig.tight_layout()
    fig.savefig(REPORTS / f"{stem}_confusion.png", dpi=160)
    plt.show()

test_summary = pd.DataFrame(test_rows)
display(test_summary)
test_summary.to_csv(REPORTS / "test_summary.csv", index=False)


## Compare with the paper carefully

The following accuracies are transcribed from **Table 5, p. 4**, and are references rather than expected outputs. The paper does not clearly identify the split for every Table 5 score. Its confusion matrix elsewhere uses the reverse axis orientation from ours. Our table keeps its reported accuracy and our measured test accuracy in distinct columns.

These are approximate matches by embedding and activation; hidden widths and pooling cannot be matched from the paper. Do not describe a numerical difference as improvement over the paper: the supplied class balance, exact split membership, PCA approximation, model details, early stopping, and checkpoint policy differ. In particular, the paper's isolated LSTM sigmoid output is ambiguous; our normalized sigmoid choice is explicit. Training-vocabulary Word2Vec (`train` mode) must be identified as a separate experiment in your report.


In [ ]:
paper = pd.DataFrame([
    ["feedforward", "word2vec_50", "relu", 0.8449],
    ["feedforward", "word2vec_100", "relu_relu", 0.8618],
    ["lstm", "word2vec_100", "h64_softmax", 0.8803],
    ["lstm", "word2vec_50", "h64_sigmoid", 0.9068],
    ["lstm", "glove_100", "h64_softmax", 0.9175],
    ["feedforward", "word2vec_50", "relu_sigmoid", 0.9183],
    ["feedforward", "glove_50", "relu_relu", 0.9486],
    ["lstm", "word2vec_50", "h64_softmax", 0.9499],
    ["feedforward", "word2vec_50", "relu_relu", 0.9572],
    ["lstm", "glove_50", "h64_softmax", 0.9586],
], columns=["family", "embedding", "config", "paper_reported_accuracy"])
reference = paper.merge(test_summary[["family", "embedding", "config", "test_accuracy"]],
                        on=["family", "embedding", "config"], how="left")
reference["word2vec_source_used"] = metadata["word2vec_source"]
display(reference)
reference.to_csv(REPORTS / "paper_reference_comparison.csv", index=False)
print("Blank measured scores mean that configuration was not selected for test evaluation.")
print("Outputs saved in:", REPORTS)


## What to include in your project write-up

Use `all_validation_results.csv` for the embedding/dimension/configuration experiment tables. Use `test_summary.csv`, the per-class classification reports, the confusion matrix PNGs, and the learning curve PNGs for evaluation. Report actual epochs and selected epochs from the validation table, not just the maximum epoch budget. Retain `artifacts/metadata.json` and each run's settings JSON with your results; these record source hashes, preparation identity, seed, versions, and assumptions.

State the supplied split sizes and class balance; pretrained versus locally trained Word2Vec; the PCA reduction if used; frozen versus trainable embeddings; the pooling and hidden widths; the validation selection metric; and early stopping status. A successful code check or a two-epoch workflow run is **not** a reproduction result. Only report the scores generated by your completed experiments.
